In [ ]:
using Pkg
Pkg.activate("TASEP")
Pkg.instantiate()
include("./modules/transcription.jl")

  Activating project at `~/Documents/UROP/TASEP/TASEP`
Precompiling project...
  ✓ Compat
  ✓ Functors
  ✓ Compat → CompatLinearAlgebraExt
  ✓ ChainRulesCore
  ✓ StaticArrayInterface
  ✓ DataStructures
  ✓ ChainRulesCore → ChainRulesCoreSparseArraysExt
  ✓ StaticArrays → StaticArraysChainRulesCoreExt
  ✓ Distances → DistancesChainRulesCoreExt
  ✓ LogExpFunctions → LogExpFunctionsChainRulesCoreExt
  ✓ SortingAlgorithms
  ✓ QuadGK
  ✓ StaticArrayInterface → StaticArrayInterfaceStaticArraysExt
  ✓ StaticArrayInterface → StaticArrayInterfaceOffsetArraysExt
  ✓ SpecialFunctions → SpecialFunctionsChainRulesCoreExt
  ✓ CloseOpenIntervals
  ✓ LayoutPointers
  ✓ StatsBase
  ✓ ComponentArrays
  ✓ StatsFuns → StatsFunsChainRulesCoreExt
  ✓ StrideArraysCore
  ✓ Graphs
  ✓ Polyester
  ✓ VertexSafeGraphs
  ✓ FastBroadcast
  ✓ RecursiveArrayTools → RecursiveArrayToolsFastBroadcastExt
  ✓ SparseDiffTools
  ✓ SciMLBase → SciMLBaseChainRulesCoreExt
  ✓ Distributions
  ✓ VectorizationBase
  ✓ Distributio

┌ Warning: Module RecursiveArrayTools with build ID fafbfcfd-2954-7dc1-0003-fad523f2e67c is missing from the cache.
│ This may mean RecursiveArrayTools [731186ca-8d62-57ce-b412-fbd966d074cd] does not support precompilation but is imported by a module that does.
└ @ Base loading.jl:1948
[ Info: Skipping precompilation since __precompile__(false). Importing RecursiveArrayToolsForwardDiffExt [14203109-85fb-5f77-af23-1cb7d9032242].
[ Info: Precompiling PreallocationTools [d236fae5-4411-538c-8e31-a6e3d9e00b46]
┌ Warning: Module ForwardDiff with build ID fafbfcfd-4719-83d4-0003-faaa41bebc79 is missing from the cache.
│ This may mean ForwardDiff [f6369f11-7733-5829-9624-2563aa707210] does not support precompilation but is imported by a module that does.
└ @ Base loading.jl:1948
[ Info: Skipping precompilation since __precompile__(false). Importing PreallocationTools [d236fae5-4411-538c-8e31-a6e3d9e00b46].
Precompiling SciMLBase
  ✓ StaticArrayInterface → StaticArrayInterfaceStaticArraysExt
  

[ Info: Precompiling Optim [429524aa-4258-5aef-a3af-852621145aeb]
┌ Warning: Module NLSolversBase with build ID fafbfcfd-9805-6b9e-0003-faaa30e91309 is missing from the cache.
│ This may mean NLSolversBase [d41bc354-129a-5804-8e4c-c37616107c6c] does not support precompilation but is imported by a module that does.
└ @ Base loading.jl:1948
[ Info: Skipping precompilation since __precompile__(false). Importing Optim [429524aa-4258-5aef-a3af-852621145aeb].
[ Info: Precompiling MathOptInterface [b8f27783-ece8-5eb3-8dc8-9495eed66fee]
┌ Warning: Module ForwardDiff with build ID fafbfcfd-4719-83d4-0003-faaa41bebc79 is missing from the cache.
│ This may mean ForwardDiff [f6369f11-7733-5829-9624-2563aa707210] does not support precompilation but is imported by a module that does.
└ @ Base loading.jl:1948
[ Info: Skipping precompilation since __precompile__(false). Importing MathOptInterface [b8f27783-ece8-5eb3-8dc8-9495eed66fee].
[ Info: Precompiling DistributionsTestExt [ffbe0ea5-a612-5ff7-aaf5

In [ ]:
#Basic parameters (from Stover et al, 2023)
k_on = 10^2.30 #P-DNA binding (hours^-1 nM^-1)
k_off = 10^3.74 #P-DNA binding (inverse hours)
k_i = 10^3.61 #Initiation rate (inverse hours)
k_e = 10^5.20 #Elongation rate (inverse hours)

#Basic pausing parameters (Klumpp and Hwa, 2008)
τ = 1/3600 #Pausing time (hours)
f = 0.1*3600 #Pausing frequency (inverse hours)

#Basic T7 RNA polymerase size parameter (My best guess + Klumpp and Hwa, 2008)
L = 25

#Basic sequence parameter 
N_all = 2117;

In [ ]:
##### Composite parameters (see overleaf derivation)
println("k_i = "*string(round(k_i; digits = 2))*" /hr")

α = 1+N_all* k_i/k_e     #Dimless
println("α = "*string(round(α; digits = 2)))

K = (k_off+k_i)/k_on     #nM
println("K = "*string(round(K; digits = 2))*" nM")

β = L*k_i^2/(k_e*k_on)   #Dimless
println("β = "*string(round(β; digits = 2)))

θ = k_off/k_on;          #nM
println("θ = "*string(round(θ; digits = 2))*" nM")

γ = k_i*f*τ^2/(1+f*τ)    #Dimless
println("γ = "*string(round(γ; digits = 2)))

p_initlimited = ComponentVector((k_i = k_i, K = K))
p_initelongation = ComponentVector((k_i = k_i, K = K, α = α))
p_basicTASEP = ComponentVector((k_i = k_i, K = K, α = α, β = β))
p_longTASEP = ComponentVector((k_i = k_i, K = K, α = α, β = β, γ = γ, θ = θ));

In [ ]:
initiationlimited = InitiationLimitedModel()
initiationelongation = InitiationElongationModel()
basicTASEP = BasicTASEPModel()
longTASEP = LPTASEPModel();

In [ ]:
p_base = ComponentVector((k_on = k_on, k_off = k_off, k_i = k_i, k_e = k_e, L = L, f = f, τ = τ, N_all = N_all))
p_composite = ComponentVector((k_i = k_i, α = α, γ = γ, L = L, N_all = N_all, k_off = k_off, k_on = k_on))

plott = 0.1
plotD = 10
plotP = 300;

In [ ]:
# plt = plot()
# plottemporaloutput!(plt, basicTASEP, p_base, plotD, plotP, 0.05)

In [ ]:
# continuumanimation(longTASEP, p_base, plotD,300, plott)

In [ ]:
# plt = densitydistributionfinal(basicTASEP, p_base, 10,100, plotlabel = "T7RNAP = 100")
# densitydistributionfinal!(plt,basicTASEP, p_base, 10,200, plotlabel = "T7RNAP = 200")
# densitydistributionfinal!(plt,basicTASEP, p_base, 10,300, plotlabel = "T7RNAP = 300")
# densitydistributionfinal!(plt,basicTASEP, p_base, 10,500, plotlabel = "T7RNAP = 500")
# densitydistributionfinal!(plt,basicTASEP, p_base, 10,1000, plotlabel = "T7RNAP = 1000")
# savefig(plt,"./figures/finalhist.png")
# plt

<h1>Showing that Analytic Approximations are Legitimate in Studied Range<h1>

In [ ]:
# Pmax = 100
# Pmin = 1
# transcriptionmodel = basicTASEP
# npoints = 100
# plt = plotQSrate_constantD(transcriptionmodel, p_composite, 1, (Pmin,Pmax);npoints = npoints, modellabel = "1 nM DNA, Analytic", color = :black)
# plotQSrate_constantD!(plt, transcriptionmodel, p_composite, 5, (Pmin,Pmax);npoints = npoints, modellabel = "5 nM DNA, Analytic", color = :black)
# plotQSrate_constantD!(plt, transcriptionmodel, p_composite, 10, (Pmin,Pmax);npoints = npoints, modellabel = "10 nM DNA, Analytic", color = :black)
# plotQSrate_constantD!(plt, transcriptionmodel, p_composite, 50, (Pmin,Pmax);npoints = npoints, modellabel = "50 nM DNA, Analytic", color = :black)
# plotQSrate_constantD!(plt, transcriptionmodel, p_composite, 100, (Pmin,Pmax);npoints = npoints, modellabel = "100 nM DNA, Analytic", color = :black)

# plotQSrate_constantD!(plt, transcriptionmodel, p_base, 1, (Pmin,Pmax);npoints = npoints, modellabel = "1 nM DNA, Direct", color = :red, analytic=false)
# plotQSrate_constantD!(plt, transcriptionmodel, p_base, 5, (Pmin,Pmax);npoints = npoints, modellabel = "5 nM DNA, Direct", color = :red, analytic=false)
# plotQSrate_constantD!(plt, transcriptionmodel, p_base, 10, (Pmin,Pmax);npoints = npoints, modellabel = "10 nM DNA, Direct", color = :red, analytic=false)
# plotQSrate_constantD!(plt, transcriptionmodel, p_base, 50, (Pmin,Pmax);npoints = npoints, modellabel = "50 nM DNA, Direct", color = :red, analytic=false)
# plotQSrate_constantD!(plt, transcriptionmodel, p_base, 100, (Pmin,Pmax);npoints = npoints, modellabel = "100 nM DNA, Direct", color = :red, analytic=false)
# savefig(plt,"./figures/comparingapproxbasicTASEP.png")
# plt

In [ ]:
# Pmax = 100
# Pmin = 1
# transcriptionmodel = longTASEP
# npoints = 100
# plt = plotQSrate_constantD(transcriptionmodel, p_composite, 1, (Pmin,Pmax);npoints = npoints, modellabel = "1 nM DNA, Analytic", color = :black)
# plotQSrate_constantD!(plt, transcriptionmodel, p_composite, 5, (Pmin,Pmax);npoints = npoints, modellabel = "5 nM DNA, Analytic", color = :black)
# plotQSrate_constantD!(plt, transcriptionmodel, p_composite, 10, (Pmin,Pmax);npoints = npoints, modellabel = "10 nM DNA, Analytic", color = :black)
# plotQSrate_constantD!(plt, transcriptionmodel, p_composite, 50, (Pmin,Pmax);npoints = npoints, modellabel = "50 nM DNA, Analytic", color = :black)
# plotQSrate_constantD!(plt, transcriptionmodel, p_composite, 100, (Pmin,Pmax);npoints = npoints, modellabel = "100 nM DNA, Analytic", color = :black)

# plotQSrate_constantD!(plt, transcriptionmodel, p_base, 1, (Pmin,Pmax);npoints = npoints, modellabel = "1 nM DNA, Direct", color = :red, analytic=false)
# plotQSrate_constantD!(plt, transcriptionmodel, p_base, 5, (Pmin,Pmax);npoints = npoints, modellabel = "5 nM DNA, Direct", color = :red, analytic=false)
# plotQSrate_constantD!(plt, transcriptionmodel, p_base, 10, (Pmin,Pmax);npoints = npoints, modellabel = "10 nM DNA, Direct", color = :red, analytic=false)
# plotQSrate_constantD!(plt, transcriptionmodel, p_base, 50, (Pmin,Pmax);npoints = npoints, modellabel = "50 nM DNA, Direct", color = :red, analytic=false)
# plotQSrate_constantD!(plt, transcriptionmodel, p_base, 100, (Pmin,Pmax);npoints = npoints, modellabel = "100 nM DNA, Direct", color = :red, analytic=false)
# savefig(plt,"./figures/comparingapproxlongTASEP.png")
# plt

<h1>Comparing Heirachies of Model Complexity<h1>

In [ ]:
# transcriptionmodel = initiationlimited
# npoints = 100
# normalizexaxis = true
# normalizeyaxis = true

# DNApoints = [2,5,5,10,50,100]
# Ppoints = [200]
# DPinputs = generategridinputs(DNApoints,Ppoints)

# modellist = [initiationlimited,initiationelongation,basicTASEP,longTASEP]
# modelnamelist = ["Initiation limited","Initiation-elongation","Basic TASEP","Long Pause TASEP"]
# modelcolorlist = [:Blues,:Greens,:Purples,:OrRd]

# plt = plot(xlabel = "T7 RNA Polymerase per DNA", ylabel = "Transcription Rate per (mM NTP/ hr) per nM DNA")

# for (jind,model) in enumerate(modellist)
#     plotmodelprediction!(plt,model,p_composite,DPinputs,normalizexaxis = normalizexaxis,normalizeyaxis =normalizeyaxis, color =modelcolorlist[jind], modelname= modelnamelist[jind])
# end
# plot!(legend = :outerright, size = (800,500),leftmargin = 5mm, bottommargin = 5mm, topmargin = 10mm)
# savefig(plt,"./figures/comparingallmodels_normalized.png")
# plt

<h1>Generating Synthetic Data for LP TASEP Model<h1>

In [ ]:
testmodelindex = 2#1  = initiation limited,  2 = initiation-elongation

DPinputs = [[0.0122/4, 24] [0.0122/2, 24] [0.0122, 24] [0.0244, 24] [0.0122, 6] [0.00305, 6] [0.0061, 6] [0.00122, 48] [0.0244, 48]]'

experimentaldata = [4.0,8.8,16.0,16.1,8.0,0.8,2.0,9.2,16.0]

###########You don't have to change anything below this line##################
DPinputs[:,1] .= DPinputs[:,1].* 1e9 ./(N_all*607.4)#DNA concentration conversion
DPinputs[:,2] .= DPinputs[:,2].*16#T7 concentration conversion

truemodelindex = 1
truemodel = modellist[truemodelindex]
truemodelname = modelnamelist[truemodelindex]
testmodel = modellist[testmodelindex]
testmodelname = modelnamelist[testmodelindex]

# #DNApoints = [0.0122, 0.0244,0.0488,0.0976] .* 1e9 ./(N_all*607.4)
# DNApoints = [0.0122/4,0.0122/2,0.0122] .* 1e9 ./(N_all*607.4)

# #Ppoints = [3,6,12, 24, 48]*16
# Ppoints = [24]*16

# DPinputs = generategridinputs(DNApoints,Ppoints)

experimentalnoise = 1e-0
npoints = 1000
normalizexaxis = false
normalizeyaxis = false

#(plt,syntheticdata,DPinputs) = generateplotsyntheticdata(truemodel,p_composite,DPinputs,normalizexaxis = normalizexaxis, normalizeyaxis = normalizeyaxis, noise = experimentalnoise, modelname= truemodelname)
plt = scatter(DPinputs[:,2],experimentaldata, label = "Our Data")
(p_null,p_thintrue,truecov) = multistartoptimizeparams(truemodel, experimentaldata, DPinputs, experimentalnoise, p_base, npoints = npoints)
(p_estimated,p_thintest,testcov) = multistartoptimizeparams(testmodel, experimentaldata, DPinputs, experimentalnoise, p_base, npoints = npoints)
plotmodelprediction!(plt,testmodel,p_estimated,DPinputs,normalizexaxis = normalizexaxis,normalizeyaxis =normalizeyaxis, modelname = testmodelname, maximumT7 = 800)
display(plt)
pltelps1 = plotparametricellipse(p_thintrue,truecov,title = truemodelname)
display(pltelps1)
pltelps2 = plotparametricellipse(p_thintest,testcov,title = testmodelname)
display(pltelps2)

In [ ]:
# a = DPinputs[1:2,:][:]
# (minf,minx,ret) = localMBDOEoptimization(testmodel,p_thintest, a, p_base,experimentalnoise,priormatrix = inv(testcov))
# println(minf)
# mbdoeDP = reshape(minx,(2,2))

In [ ]:
# function Doptimalitywrapper(model,estimatedparams, DPpointsflattened, baseparams, noise, grad; kwargs...)
#     if length(grad) > 0
#         fun = x->getDoptimalitymeasure(model,estimatedparams, x, baseparams, noise; kwargs...)
#         gnew = ForwardDiff.gradient(fun,DPpointsflattened)
#         for i in 1:length(grad)
#             grad[i] = gnew[i]
#         end
#     end
#     dmeasure = getDoptimalitymeasure(model,estimatedparams, DPpointsflattened, baseparams, noise; kwargs...)
#     return dmeasure
# end

In [ ]:
# function localMBDOEoptimization(model, params, initDPpoints, baseparams, noise; upperDNA = 75.0, lowerDNA = 1.0, upperP = 800.0, lowerP = 40.0, priormatrix = zeros(length(params),length(params)))
#     nvariables = Int(length(initDPpoints)/2)
#     opt = Opt(:LD_SLSQP, nvariables*2)
#     opt.lower_bounds = vcat(ones(nvariables)*lowerDNA,ones(nvariables)*lowerP)
#     opt.upper_bounds = vcat(ones(nvariables)*upperDNA,ones(nvariables)*upperP)
#     opt.ftol_rel = 1e-10
#     opt.maxtime = 1
#     opt.min_objective = (x,g) -> Doptimalitywrapper(model, params, x, baseparams, noise, g, priormatrix = priormatrix)
#     (minf,minx,ret) = optimize(opt, initDPpoints)
#     return (minf,minx,ret)
# end